# 额外周末练习 — 第 2 周

## 练习目标

运用第 2 周学到的一切，为第 1 周练习中构建的技术问答器做一个完整原型。

应包括：

- **Gradio UI**
- **流式输出**（`stream=True`，边生成边更新 Chatbot）
- 用 **系统提示词** 增加专业性
- **模型切换**（OpenRouter 上的 GPT / Gemini）
- （加分）工具使用
- （更大胆）**音频输入** + **音频回复**（本笔记本：Whisper 转写 + Groq Orpheus TTS）

商业想象空间很大：语言导师、入职培训、课程伴学 AI 等。

## 怎么跑

1. `.env` 准备：`OPENROUTER_API_KEY`、`HF_TOKEN`、`GROQ_API_KEY`
2. 依次运行单元格；最后一格打开 Gradio
3. 选模型 → 用麦克风提问 → 看文字流式出现，并听语音回复


In [ ]:
# ========== 导入：OpenRouter / Gradio / Whisper / Groq TTS ==========

# 标准库 os：读环境变量、拼临时音频路径
import os
# load_dotenv：把 .env 密钥读入环境
from dotenv import load_dotenv
# OpenAI 兼容客户端：这里用来打 OpenRouter
from openai import OpenAI
# Gradio：搭 Blocks 界面
import gradio as gr
# transformers：加载 ASR pipeline（Whisper）
import transformers
from transformers import pipeline
# torch：深度学习后端（pipeline 可能用到）
import torch
# Accelerator：自动选择 CUDA / MPS / CPU 设备
from accelerate import Accelerator
# Hugging Face 登录（若模型需鉴权可用）
from huggingface_hub import login
# Groq 客户端：本练习用其 TTS（Orpheus）
from groq import Groq
# tempfile：把合成语音写到系统临时目录
import tempfile


In [ ]:
# ========== 加载并检查环境变量 ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)
# OpenRouter 的 API Key（走 OpenAI 兼容 base_url）
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
# Hugging Face token：拉 Whisper 等模型时可能需要
hf_token = os.getenv("HF_TOKEN")
# Groq API Key：文本转语音
groq_api_key = os.getenv("GROQ_API_KEY")

# 逐项打印是否配置成功（不打印完整密钥内容）
if openrouter_api_key:
    print("OPENROUTER_API_KEY is set.")
else:
    print("OPENROUTER_API_KEY is not set.")

if hf_token:
    print("HuggingFace token found.")
else:
    print("No HuggingFace token found.")

if groq_api_key:
    print("GROQ_API_KEY is set.")
else:
    print("GROQ_API_KEY is not set.")


In [ ]:
# ========== 常量：OpenRouter 上的模型 id 与网关地址 ==========

# OpenRouter 风格的模型名：提供方/模型
MODEL_GPT = 'openai/gpt-4o-mini'
MODEL_GEMINI = 'google/gemini-2.5-flash-lite'
# OpenRouter OpenAI 兼容 API 根路径（字符串勿改）
openrouter_url = "https://openrouter.ai/api/v1"

# Dropdown 可选模型列表
models = [MODEL_GPT, MODEL_GEMINI]


In [ ]:
# ========== 创建两个客户端：LLM（OpenRouter）与 TTS（Groq） ==========

# base_url 指向 OpenRouter；api_key 用 OPENROUTER_API_KEY
client_llm = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
# Groq 专用于后续 text_to_audio
client_groq = Groq(api_key=groq_api_key)


In [ ]:
# ========== 系统提示词：技术问答助手的风格约束 ==========
# 英文 prompt 保持原样——它直接影响模型回答行为

system_prompt = """ 
You are a technical assistant.
Your task is to take a technical question and produce a clear, accurate, and well-structured explanation.
Guidelines:
- Prioritize clarity over complexity.
- If the question lacks necessary details, state assumptions clearly.
- Avoid fluff, marketing language, or unnecessary verbosity.
- Respond in one concise paragraph, using simple language and examples when helpful
"""


In [ ]:
# ========== 把转写得到的问题，包装成 user prompt ==========

def get_user_prompt(question):
    # f-string 嵌入用户问题；提示词英文保持原样
    user_prompt = f"""
    You are a technical assistant. 
    Please answer the following question in a clear, concise, and structured manner, following the guidelines provided.
    Question: {question}
    """
    return user_prompt


In [ ]:
# ========== 设备选择：用 Accelerator 自动挑 GPU/CPU ==========

device = Accelerator().device


In [ ]:
# ========== 加载 Whisper：把麦克风音频转成英文文本 ==========

transcriber = pipeline(
    "automatic-speech-recognition",
    # 英文小模型，速度快；model id 保持原样
    model="openai/whisper-base.en",
    # 放到 Accelerator 选中的设备上
    device=device
    )


In [ ]:
# ========== TTS：文本 → 音频文件路径（Groq Orpheus） ==========

def text_to_audio(text,
                  model="canopylabs/orpheus-v1-english",
                  voice="troy",
                  response_format="wav",
                  ):
    """
    使用 Groq Orpheus API 把输入文本合成语音。
    返回生成的音频文件路径。
    """
    # 调用 Groq speech API；参数名/默认值保持原样
    response = client_groq.audio.speech.create(
        model=model,
        voice=voice,
        input=text,
        response_format=response_format
    )

    # 写到系统临时目录，例如 /tmp/output.wav
    output_path = os.path.join(tempfile.gettempdir(), f"output.{response_format}")
    response.write_to_file(output_path)
    return output_path


In [ ]:
# ========== 主流程：音频 → 转写 → 流式 LLM → TTS ==========

def process_input(audio_file,history,model):
    # 规范化 Gradio history 为 role/content 列表
    history = [{"role":h["role"], "content":h["content"]} for h in history]

    # Whisper 转写麦克风文件，取出 text 字段
    transcription = transcriber(audio_file)["text"]
    # 包装成带指南的 user prompt
    prompt = get_user_prompt(transcription)
    # messages = system + 历史 + 当前用户
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": prompt}]
    # OpenRouter 流式聊天；model 来自 Dropdown
    stream = client_llm.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    # 先把本轮 user/assistant 占位写进 history，便于边生成边 yield 更新 UI
    history.append({"role": "user", "content": transcription})
    history.append({"role": "assistant", "content": ""})

    full_response = ""
    for chunk in stream:
        # delta.content 可能为 None，用 or "" 兜底
        delta = chunk.choices[0].delta.content or ""
        full_response += delta
        # 不断刷新最后一条 assistant 内容，实现「打字机」流式
        history[-1]["content"] = full_response
        # 流式阶段先不返回音频（None）
        yield history, None

    print("LLM Response: ", full_response)
    try:
        # 全文生成完后再 TTS
        output_audio = text_to_audio(full_response)
    except Exception as e:
        # TTS 失败不拖垮主流程
        print("Error in text-to-audio conversion: ", e)
        output_audio = None

    # 最终一次返回：完整 history + 音频路径
    return history, output_audio


In [ ]:
# ========== Gradio Blocks：选模型 + 录问题 + 听回答 ==========

with gr.Blocks() as demo:
    # 界面标题（Gradio Markdown；英文 UI 文案可保留）
    gr.Markdown("# Technical Q&A Assistant with Audio")
    with gr.Row():
        chatbot = gr.Chatbot(height=400)
    with gr.Row():
        # 下拉切换 OpenRouter 上的两个模型
        model = gr.Dropdown(
            choices=models,
            value=models[0],
            label="Select Model",
            interactive=True
            )

    with gr.Row():
        with gr.Column(scale=6):
            # 麦克风录音；filepath 交给 Whisper
            audio_input = gr.Audio(sources="microphone", type="filepath",label="Record your question")
            submit_btn = gr.Button("Submit Audio")
        with gr.Column(scale=6):
            # streaming/autoplay：方便播放最终 TTS
            audio_output = gr.Audio(label="Audio Response", streaming=True, autoplay=True)
    # 点击提交：跑 process_input（生成器会多次更新 outputs）
    submit_btn.click(fn=process_input, inputs=[audio_input,chatbot,model], outputs=[chatbot,audio_output])

# 启动并尝试打开浏览器
demo.launch(inbrowser=True)
